In [2]:
import sys
sys.path.append('/n/home06/drooryck/codeswitching-llms')
from pathlib import Path
from july_aug_sept_exp.src.metrics import Metrics
from july_aug_sept_exp.src.dataset_manager import DatasetManager
from july_aug_sept_exp.src.model_config import ModelConfig, SlurmConfig
from july_aug_sept_exp.src.experiment import Experiment
from july_aug_sept_exp.src.plotting import BilingualPlotter

/n/home06/drooryck/envs/codeswitching-py310/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [6]:
repo_root = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/")
output_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/oct21.3")
#output_dir = Path("/n/netscratch/dam_lab/Lab/drooryck/codeswitching-llms/results/oct21_overnight")
output_dir.mkdir(parents=True, exist_ok=True)

config = ModelConfig.load(repo_root / "configs" / "default_model.json")
slurm_config = SlurmConfig.load(repo_root / "configs" / "slurm_default.json")
slurm_config.output_pattern = str(output_dir / "logs" / "slurm_%A_%a.out")
slurm_config.error_pattern = str(output_dir / "logs" / "slurm_%A_%a.err")
slurm_config.job_name = "oct21.3exp" 

data_manager = DatasetManager(str(repo_root / "data"), config, lexicon_path = "/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/data/lexicon_sep22.json")
metrics = Metrics("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/data/lexicon_sep22.json")
experiment = Experiment(config, data_manager, metrics, output_dir)

config.save(output_dir / "model_config.json")
slurm_config.save(output_dir / "slurm_config.json")

# props = [0.000, 0.001, 0.005, 0.010, 0.015, 0.020, 0.025, 0.030, 
# 0.040, 0.050, 0.075, 0.100, 0.150, 0.200, 0.250, 0.300, 0.400, 0.450
# , 0.500, 0.550, 0.600, 0.650, 0.700, 0.750, 0.800, 0.850, 0.900, 0.925, 0.950,
#  0.960, 0.970, 0.980, 0.985, 0.990, 0.995, 0.999, 1.000]
# runs = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]
props = [0.1, 0.25, 0.5, 0.75, 0.9]
runs = [23, 34]

job_ids = experiment.submit_to_slurm(
    props=props,
    runs=runs,
    slurm_config=slurm_config,
    eval_prop=0.05
)

2025-10-21 04:48:22,463 |     INFO | Submitting 10 jobs to SLURM...
2025-10-21 04:48:22,534 |     INFO | Submitted job array 42449517


In [2]:
# Define the results directory (where data is)
results_dir = Path("/n/netscratch/dam_lab/Lab/drooryck/codeswitching-llms/results/oct21_overnight")

# Define plots output directory (in home)
plots_base_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_oct21")
plots_base_dir.mkdir(parents=True, exist_ok=True)

# Collect metrics from all runs
def collect_all_metrics(results_dir):
    """Collect metrics from all ablation types across all runs."""
    all_results = []
    
    # Find all run directories (p*_run*)
    run_dirs = sorted([d for d in results_dir.iterdir() if d.is_dir() and d.name.startswith('p')])
    
    print(f"Found {len(run_dirs)} run directories")
    
    for run_dir in run_dirs:
        # Parse directory name to get prop and run_id
        # Format: p00.0_run1 or p100.0_run20
        dir_name = run_dir.name
        if "_run" not in dir_name:
            continue
            
        parts = dir_name.split("_run")
        prop = float(parts[0][1:]) / 100.0  # Remove 'p' and convert to decimal
        run_id = int(parts[1])
        
        # Load metrics for each ablation type
        for ablation_type in ["none", "subject", "verb", "object"]:
            metrics_file = run_dir / f"ablation_{ablation_type}_metrics.json"
            
            if not metrics_file.exists():
                print(f"Warning: {metrics_file} not found")
                continue
            
            with open(metrics_file, 'r') as f:
                metrics = json.load(f)
            
            # Create result record
            result = {
                'prop': prop,
                'run_id': run_id,
                'ablation': ablation_type,
                'run_dir': str(run_dir),
                **metrics  # Unpack all metrics
            }
            all_results.append(result)
    
    return pd.DataFrame(all_results)

# Collect all results
print("Collecting metrics from all runs...")
results_df = collect_all_metrics(results_dir)

print(f"\nCollected {len(results_df)} metric records")
print(f"Unique proportions: {len(results_df['prop'].unique())}")
print(f"Unique run_ids: {sorted(results_df['run_id'].unique())}")
print(f"Ablation types: {sorted(results_df['ablation'].unique())}")

# Save summary to the plots directory for reference
summary_path = plots_base_dir / "all_metrics.csv"
results_df.to_csv(summary_path, index=False)
print(f"\nSaved summary to: {summary_path}")

# Create plots for each ablation type separately
for ablation_type in ["none", "subject", "verb", "object"]:
    print(f"\n{'='*60}")
    print(f"Creating plots for ablation={ablation_type}")
    print(f"{'='*60}")
    
    # Filter to this ablation type
    abl_df = results_df[results_df['ablation'] == ablation_type].copy()
    
    # Create plotter for this ablation type
    abl_plots_dir = plots_base_dir / f"ablation_{ablation_type}"
    plotter = BilingualPlotter(abl_df, abl_plots_dir)
    
    # Create all plots
    plotter.create_all_plots()
    
    print(f"✅ Plots saved to: {abl_plots_dir}")

print(f"\n{'='*60}")
print("All plots created!")
print(f"{'='*60}")
print(f"Main plots directory: {plots_base_dir}")
print(f"\nView plots at:")
print(f"  {plots_base_dir}/ablation_none/")
print(f"  {plots_base_dir}/ablation_subject/")
print(f"  {plots_base_dir}/ablation_verb/")
print(f"  {plots_base_dir}/ablation_object/")

Found 740 run directories

Collected 2960 metric records
Unique proportions: 37
Unique run_ids: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]
Ablation types: ['none', 'object', 'subject', 'verb']

Saved summary to: /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_oct21/all_metrics.csv

Creating plots for ablation=none
✅ Plots saved to: /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_oct21/ablation_none

Creating plots for ablation=subject
✅ Plots saved to: /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_oct21/ablation_subject

Creating plots for ablation=verb
✅ Plots saved to: /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_oct21/ablation_verb

Creating plots

In [ ]:
# ## get language orientation score over prediction sentences
# def language_orientation_score(
#     self,
#     pred: str,
#     ablation_type: str | None = None,
#     input_lang: str | None = None,
# ) -> float:
#     """
#     Frenchiness in [0,1]: 0 = strongly Dutch, 1 = strongly French.
#     Weights: structure(0.35) > aux(0.30) > participle(0.20) > det(0.10) > noun(0.05).
#     Missing/unrecognized cues contribute 0.5 (neutral).
#     """
#     NEU = 0.5
#     w_struct, w_aux, w_part, w_det, w_noun = 0.2, 0.2, 0.20, 0.20, 0.2

#     tokens = self.tokenize(pred.lower())

#     # --- Structure: FR=1, NL=0, neither=0.5
#     st = self.check_structure_conformity(pred)
#     if   st.get("follows_fr_structure"): s_struct = 1.0
#     elif st.get("follows_nl_structure"): s_struct = 0.0
#     else:                                s_struct = NEU

#     # --- Aux: FR=1, NL=0, missing=0.5
#     aux = next((t for t in tokens if t in (self.aux_fr | self.aux_nl)), None)
#     if   aux is None:           s_aux = NEU
#     elif aux in self.aux_fr:    s_aux = 1.0
#     elif aux in self.aux_nl:    s_aux = 0.0
#     else:                       s_aux = NEU

#     # --- Participle: FR=1, NL=0, missing=0.5
#     part = next((t for t in tokens if t in (self.part_fr | self.part_nl)), None)
#     if   part is None:          s_part = NEU
#     elif part in self.part_fr:  s_part = 1.0
#     elif part in self.part_nl:  s_part = 0.0
#     else:                       s_part = NEU

#     # --- Determiners: fraction FR among recognized dets; none -> 0.5
#     dets = [t for t in tokens if t in (self.det_fr | self.det_nl)]
#     if dets:
#         fr = sum(t in self.det_fr for t in dets)
#         nl = sum(t in self.det_nl for t in dets)
#         s_det = (fr / (fr + nl)) if (fr + nl) > 0 else NEU
#     else:
#         s_det = NEU

#     # --- Nouns: fraction FR among recognized nouns; none -> 0.5
#     nouns = [t for t in tokens if t in (self.nouns_fr | self.nouns_nl)]
#     if nouns:
#         fr = sum(t in self.nouns_fr for t in nouns)
#         nl = sum(t in self.nouns_nl for t in nouns)
#         s_noun = (fr / (fr + nl)) if (fr + nl) > 0 else NEU
#     else:
#         s_noun = NEU

#     score = (
#         w_struct * s_struct +
#         w_aux    * s_aux +
#         w_part   * s_part +
#         w_det    * s_det +
#         w_noun   * s_noun
#     )
#     return float(max(0.0, min(1.0, score)))

In [26]:
from importlib import reload
import july_aug_sept_exp.src.metrics as metrics
import july_aug_sept_exp.src.plotting as plotting

# Reload metrics first (since plotting might import metrics)
reload(metrics)
reload(plotting)

# Now get the classes
Metrics = metrics.Metrics
BilingualPlotter = plotting.BilingualPlotter

In [16]:
import pandas as pd
run_id, ablation = 2, "object"
pd.read_csv(f"/n/netscratch/dam_lab/Lab/drooryck/codeswitching-llms/results/oct21_overnight/p50.0_run{run_id}/ablation_predictions.csv").query(f"ablation == '{ablation}'").head(10)

,language,input,gold,prediction,ablation
3,fr,le chien aime de wolf,le chien a aimé le loup,de chien a aimé le wolf,object
7,fr,les chiens aiment de wolven,les chiens ont aimé les loups,de chiens ont aimé les wolven,object
11,fr,le chien frappe de wolf,le chien a frappé le loup,de chien a frappé le wolf,object
15,fr,les chiens frappent de wolven,les chiens ont frappé les loups,de chiens ont frappé les wolven,object
19,fr,le chien attaque de wolf,le chien a attaqué le loup,de chien a attaqué le wolf,object
23,fr,les chiens attaquent de wolven,les chiens ont attaqué les loups,de chiens ont attaqué les wolven,object
27,fr,le chien prend de wolf,le chien a pris le loup,de chien a pris le wolf,object
31,fr,les chiens prennent de wolven,les chiens ont pris les loups,de chiens ont pris les wolven,object
35,fr,le chien tire de wolf,le chien a tiré le loup,de chien a tiré le wolf,object
39,fr,les chiens tirent de wolven,les chiens ont tiré les loups,de chiens ont tiré les wolven,object


In [19]:
metrics = Metrics("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/data/lexicon_sep22.json")
# df = pd.read_csv("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p20.0_run4/ablation_predictions.csv")
# sample_pred = df.iloc[0]['prediction']
sample_pred = "de chien a aimé le wolf"
result = metrics.language_orientation_score(sample_pred, return_breakdown=True)

print(f"Sentence: {sample_pred}")
print(f"Orientation score: {result['score']:.3f}")  # ← Access the 'score' key
print(f"\nComponent breakdown:")
for comp, val in result['components'].items():
    print(f"  {comp:12s}: {val:.2f}")

# Test structure check separately
structure = metrics.check_structure_conformity(sample_pred)
print(f"Structure: {structure}")

Sentence: de chien a aimé le wolf
Orientation score: 0.800

Component breakdown:
  structure   : 1.00
  aux         : 1.00
  participle  : 1.00
  determiner  : 0.50
  noun        : 0.50
Structure: {'follows_fr_structure': True, 'follows_nl_structure': False, 'follows_either_structure': True, 'token_types': ['det_nl', 'noun_fr', 'aux_fr', 'part_fr', 'det_fr', 'noun_nl']}


In [21]:
# Test sentences along the frenchiness spectrum
test_sentences = [
    # Pure Dutch (score ≈ 0.0)
    ("de hond heeft de wolf gemogen", "Pure NL: structure, aux, participle, dets, nouns all NL"),
    
    # Mostly Dutch (score ≈ 0.2-0.3)
    ("de hond heeft de wolf aimé", "NL structure + FR participle only"),
    ("de chien heeft de wolf gemogen", "NL structure/aux/participle + FR nouns"),
    ("le hond heeft de wolf gemogen", "NL structure/aux/participle + 1 FR det"),
    
    # Mixed leaning Dutch (score ≈ 0.4-0.5)
    ("de chien heeft de wolf aimé", "NL structure/aux + FR participle/nouns"),
    ("le chien heeft le wolf gemogen", "NL structure/aux/participle + FR dets/nouns"),
    ("de hond heeft aimé de wolf", "NL structure + FR participle + mixed"),
    
    # Balanced mixed (score ≈ 0.5)
    ("le chien heeft aimé de wolf", "Mixed structure, FR dets/nouns/participle, NL aux"),
    ("de chien a gemogen le wolf", "Mixed everything"),
    
    # Mixed leaning French (score ≈ 0.6-0.7)
    ("le chien a gemogen le loup", "FR structure/aux/dets/nouns + NL participle"),
    ("le chien a aimé de wolf", "FR structure/aux/participle/dets + NL nouns"),
    ("le chien a aimé de loup", "FR structure/aux/participle + 1 NL det"),
    
    # Mostly French (score ≈ 0.8-0.9)
    ("le chien a aimé le wolf", "FR structure/aux/participle/dets + 1 NL noun"),
    
    # Pure French (score ≈ 1.0)
    ("le chien a aimé le loup", "Pure FR: all components French"),
]

# Test them
from july_aug_sept_exp.src.metrics import Metrics

metrics = Metrics("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/data/lexicon_sep22.json")

print("Sentence                              | Score | Description")
print("-" * 90)
for sentence, description in test_sentences:
    result = metrics.language_orientation_score(sentence, return_breakdown=True)
    score = result['score']
    print(f"{sentence:38s} | {score:.3f} | {description}")

# Show detailed breakdown for one example
print("\n" + "="*90)
result = metrics.language_orientation_score("de chien heeft de wolf gemogen", return_breakdown=True)
print(f"Overall score: {result['score']:.3f}")
for comp, val in result['components'].items():
    print(f"  {comp:12s}: {val:.2f}")

Sentence                              | Score | Description
------------------------------------------------------------------------------------------
de hond heeft de wolf gemogen          | 0.000 | Pure NL: structure, aux, participle, dets, nouns all NL
de hond heeft de wolf aimé             | 0.200 | NL structure + FR participle only
de chien heeft de wolf gemogen         | 0.100 | NL structure/aux/participle + FR nouns
le hond heeft de wolf gemogen          | 0.100 | NL structure/aux/participle + 1 FR det
de chien heeft de wolf aimé            | 0.300 | NL structure/aux + FR participle/nouns
le chien heeft le wolf gemogen         | 0.300 | NL structure/aux/participle + FR dets/nouns
de hond heeft aimé de wolf             | 0.400 | NL structure + FR participle + mixed
le chien heeft aimé de wolf            | 0.600 | Mixed structure, FR dets/nouns/participle, NL aux
de chien a gemogen le wolf             | 0.600 | Mixed everything
le chien a gemogen le loup             | 0.800 | FR s

In [ ]:
## creating plots for sep24.3 runs


# Setup
results_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3")
plots_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_sep24/orientation")
plots_dir.mkdir(parents=True, exist_ok=True)

metrics_obj = Metrics("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/data/lexicon_sep22.json")

# Compute orientation scores from predictions
orientation_results = []
run_dirs = sorted([d for d in results_dir.iterdir() if d.is_dir() and d.name.startswith('p')])

for run_dir in run_dirs:
    print('starting run', run_dir)
    parts = run_dir.name.split("_run")
    prop = float(parts[0][1:]) / 100.0
    run_id = int(parts[1])
    
    pred_file = run_dir / "ablation_predictions.csv"
    if not pred_file.exists():
        continue
    
    df = pd.read_csv(pred_file)
    df_none = df[df['ablation'] == 'none']  # Only non-ablated
    
    # Compute average orientation for each language
    for lang in ['fr', 'nl']:
        lang_preds = df_none[df_none['language'] == lang]['prediction']
        scores = [metrics_obj.language_orientation_score(pred) for pred in lang_preds]
        avg_score = sum(scores) / len(scores) if scores else 0.5
        
        orientation_results.append({
            'prop': prop,
            'run_id': run_id,
            f'{lang}_orientation': avg_score
        })



starting run /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p00.0_run1
starting run /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p00.0_run2
starting run /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p00.0_run3
starting run /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p00.0_run4
starting run /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p00.0_run5
starting run /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p00.1_run1
starting run /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p00.1_run2
starting run /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p00.1_run3
starting run /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p00.1_run4
starting run /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p00.1_run5
starting run /n/home06/drooryck/codeswit

ValueError: could not convert string to float: 'lots_none'

In [ ]:
# from pathlib import Path
# import pandas as pd
# from july_aug_sept_exp.src.metrics import Metrics
# from july_aug_sept_exp.src.plotting import BilingualPlotter
# from joblib import Parallel, delayed

# # Setup
# results_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3")
# base_plots_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_sep24")

# run_dirs = sorted([d for d in results_dir.iterdir() if d.is_dir() and d.name.startswith('p')])

# def process_run(run_dir, ablation_type):
#     """Process a single run directory for orientation scores."""
#     print('starting run', run_dir)
#     metrics_obj = Metrics("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/data/lexicon_sep22.json")
    
#     parts = run_dir.name.split("_run")
#     prop = float(parts[0][1:]) / 100.0
#     run_id = int(parts[1])
    
#     pred_file = run_dir / "ablation_predictions.csv"
#     if not pred_file.exists():
#         return []
    
#     df = pd.read_csv(pred_file)
#     df_abl = df[df['ablation'] == ablation_type]
    
#     results = []
#     for lang in ['fr', 'nl']:
#         lang_preds = df_abl[df_abl['language'] == lang]['prediction']
#         scores = [metrics_obj.language_orientation_score(pred) for pred in lang_preds]
#         avg_score = sum(scores) / len(scores) if scores else 0.5
        
#         results.append({
#             'prop': prop,
#             'run_id': run_id,
#             f'{lang}_orientation': avg_score
#         })
#     return results

# # Process each ablation type
# for ablation_type in ['none', 'subject', 'verb', 'object']:
#     print(f"\nProcessing ablation={ablation_type}...")
    
#     # Parallel processing across all runs
#     all_results = Parallel(n_jobs=8)(
#         delayed(process_run)(run_dir, ablation_type) 
#         for run_dir in run_dirs
#     )
    
#     # Flatten results
#     orientation_results = [item for sublist in all_results for item in sublist]
    
#     # Create DataFrame
#     orientation_df = pd.DataFrame(orientation_results)
#     orientation_df = orientation_df.groupby(['prop', 'run_id']).first().reset_index()
    
#     # Create plot
#     plots_dir = base_plots_dir / f"ablation_{ablation_type}"
#     plots_dir.mkdir(parents=True, exist_ok=True)
    
#     plotter = BilingualPlotter(orientation_df, plots_dir)
#     plotter.plot_language_orientation()
    
#     print(f"  ✓ Saved to {plots_dir / 'orientation.png'}")

# print(f"\nDone! All orientation plots saved to {base_plots_dir}")


Processing ablation=none...


: 

In [31]:
orientation_results

[{'prop': 0.0, 'run_id': 1, 'fr_orientation': 0.016236210253083486},
 {'prop': 0.0, 'run_id': 1, 'nl_orientation': 0.0},
 {'prop': 0.0, 'run_id': 10, 'fr_orientation': 0.005317975340687715},
 {'prop': 0.0, 'run_id': 10, 'nl_orientation': 0.0},
 {'prop': 0.0, 'run_id': 11, 'fr_orientation': 0.050762491888379904},
 {'prop': 0.0, 'run_id': 11, 'nl_orientation': 0.0}]

In [30]:
## creating plots for oct21_overnight runs
import importlib
from july_aug_sept_exp.src import plotting
importlib.reload(plotting)

from pathlib import Path
import pandas as pd
from july_aug_sept_exp.src.metrics import Metrics
from july_aug_sept_exp.src.plotting import BilingualPlotter

# Setup for netscratch data
results_dir = Path("/n/netscratch/dam_lab/Lab/drooryck/codeswitching-llms/results/oct21_overnight")
plots_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_oct21_overnight/orientation")
plots_dir.mkdir(parents=True, exist_ok=True)

metrics_obj = Metrics("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/data/lexicon_sep22.json")

# Compute orientation scores from predictions
orientation_results = []
run_dirs = sorted([d for d in results_dir.iterdir() if d.is_dir() and d.name.startswith('p')])

print(f"Processing {len(run_dirs)} runs...")
for run_dir in run_dirs:
    parts = run_dir.name.split("_run")
    prop = float(parts[0][1:]) / 100.0
    run_id = int(parts[1])
    
    pred_file = run_dir / "ablation_predictions.csv"
    if not pred_file.exists():
        continue
    
    df = pd.read_csv(pred_file)
    df_none = df[df['ablation'] == 'none']
    
    # Compute average orientation for each language
    for lang in ['fr', 'nl']:
        lang_preds = df_none[df_none['language'] == lang]['prediction']
        scores = [metrics_obj.language_orientation_score(pred) for pred in lang_preds]
        avg_score = sum(scores) / len(scores) if scores else 0.5
        
        orientation_results.append({
            'prop': prop,
            'run_id': run_id,
            f'{lang}_orientation': avg_score
        })

# Create DataFrame
orientation_df = pd.DataFrame(orientation_results)
orientation_df = orientation_df.groupby(['prop', 'run_id']).first().reset_index()

# Create plot
plotter = BilingualPlotter(orientation_df, plots_dir)
plotter.plot_language_orientation()

print(f"Plot saved to {plots_dir / 'orientation.png'}")


Processing 740 runs...


KeyboardInterrupt: 

# OCT 23:

In [4]:
# Jupyter cell to create orientation plots from pre-computed oct21_overnight data
# Reads from temp_plots_oct21_overnight/ and creates plots similar to your reference

import sys
sys.path.insert(0, '/n/home06/drooryck/codeswitching-llms')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setup paths - read from pre-computed data
base_plots_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_oct21_overnight")
output_plots_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_oct21_orientation")
output_plots_dir.mkdir(parents=True, exist_ok=True)

print(f"Reading pre-computed orientation scores from {base_plots_dir}")

# Process each ablation type
for ablation_type in ['none', 'subject', 'verb', 'object']:
    print(f"\nProcessing ablation: {ablation_type}")
    
    # Read pre-computed orientation scores
    csv_path = base_plots_dir / f"ablation_{ablation_type}" / "orientation_scores.csv"
    if not csv_path.exists():
        print(f"No data found for ablation {ablation_type}")
        continue
    
    df = pd.read_csv(csv_path)
    print(f"Loaded DataFrame with {len(df)} rows")
    
    # Create the plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Get unique run_ids for color mapping
    unique_runs = df['run_id'].unique()
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique_runs)))  # Generate colors
    
    # Plot French
    fr_data = df[df['language'] == 'fr'] if 'language' in df.columns else df
    for i, run_id in enumerate(fr_data['run_id'].unique()):
        run_data = fr_data[fr_data['run_id'] == run_id].sort_values('prop')
        ax1.plot(run_data['prop'], run_data['fr_orientation'], 
                alpha=0.7, linewidth=1.5, color=colors[i % len(colors)])
    
    # Add median line
    medians = fr_data.groupby('prop')['fr_orientation'].median()
    ax1.plot(medians.index, medians.values, 
            color='black', linewidth=4, label='Median', linestyle='--')
    
    # Add mean line
    means = fr_data.groupby('prop')['fr_orientation'].mean()
    ax1.plot(means.index, means.values, 
            color='red', linewidth=3, label='Mean', linestyle=':')
    
    ax1.set_title(f'French Test Sentences\n({ablation_type} ablation)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Proportion of French in Training Data')
    ax1.set_ylabel('Language Orientation (0=Dutch, 1=French)')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1)
    ax1.legend()
    
    # Plot Dutch
    nl_data = df[df['language'] == 'nl'] if 'language' in df.columns else df
    for i, run_id in enumerate(nl_data['run_id'].unique()):
        run_data = nl_data[nl_data['run_id'] == run_id].sort_values('prop')
        ax2.plot(run_data['prop'], run_data['nl_orientation'], 
                alpha=0.7, linewidth=1.5, color=colors[i % len(colors)])
    
    # Add median line
    medians = nl_data.groupby('prop')['nl_orientation'].median()
    ax2.plot(medians.index, medians.values, 
            color='black', linewidth=4, label='Median', linestyle='--')
    
    # Add mean line
    means = nl_data.groupby('prop')['nl_orientation'].mean()
    ax2.plot(means.index, means.values, 
            color='red', linewidth=3, label='Mean', linestyle=':')
    
    ax2.set_title(f'Dutch Test Sentences\n({ablation_type} ablation)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Proportion of French in Training Data')
    ax2.set_ylabel('Language Orientation (0=Dutch, 1=French)')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1)
    ax2.legend()
    
    # Overall title
    n_runs = len(fr_data['run_id'].unique())
    fig.suptitle(f'Language Orientation Score - {ablation_type.title()} Ablation\n({n_runs} runs per proportion)', 
                fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    
    # Save plot
    output_path = output_plots_dir / f'orientation_line_{ablation_type}.png'
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    print(f"Saved plot to {output_path}")

print(f"\nAll plots saved to {output_plots_dir}")

Reading pre-computed orientation scores from /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_oct21_overnight

Processing ablation: none
Loaded DataFrame with 740 rows
Saved plot to /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_oct21_orientation/orientation_line_none.png

Processing ablation: subject
Loaded DataFrame with 740 rows
Saved plot to /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_oct21_orientation/orientation_line_subject.png

Processing ablation: verb
Loaded DataFrame with 740 rows
Saved plot to /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_oct21_orientation/orientation_line_verb.png

Processing ablation: object
Loaded DataFrame with 740 rows
Saved plot to /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_oct21_orientation/orientation_line_object.png

All plots saved to /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_p

### testing if data handling

In [27]:
# Create DataFrame (one row per run with both fr_orientation and nl_orientation)
orientation_df = pd.DataFrame(orientation_results)
orientation_df = orientation_df.groupby(['prop', 'run_id']).first().reset_index()

# Create plot using BilingualPlotter
plotter = BilingualPlotter(orientation_df, plots_dir)
plotter.plot_language_orientation()

print(f"Plot saved to {plots_dir / 'orientation.png'}")

Plot saved to /n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/scripts/temp_plots_sep24/orientation/orientation.png


In [1]:
import sys
sys.path.insert(0, '/n/home06/drooryck/codeswitching-llms')

import pandas as pd
import numpy as np
from pathlib import Path

# Simulate the nested sampling logic
def simulate_sampling(train_df_full, prop, run_id, eval_prop=0.05):
    """Simulate the sampling logic from experiment.py"""
    rng = np.random.RandomState(run_id)
    train_df_full = train_df_full.copy()
    train_df_full["global_key"] = rng.random(len(train_df_full))
    train_df_full["orig_idx"] = np.arange(len(train_df_full))
    
    # Validation split
    eval_mask = train_df_full["global_key"] < eval_prop
    val_df = train_df_full[eval_mask].reset_index(drop=True)
    train_df_remaining = train_df_full[~eval_mask].reset_index(drop=True)
    
    # Language ranking
    train_df_remaining["lang_rank"] = (
        train_df_remaining.groupby("lang")["global_key"]
        .rank(method="first")
        .astype(int)
    )
    
    # Budget calculation
    total_budget = min(
        (train_df_remaining.lang == "fr").sum(),
        (train_df_remaining.lang == "nl").sum()
    )
    
    want_fr = int(total_budget * prop)
    want_nl = total_budget - want_fr
    
    # Nested selection
    fr_take = train_df_remaining[
        (train_df_remaining.lang == "fr") & (train_df_remaining.lang_rank <= want_fr)
    ]
    nl_take = train_df_remaining[
        (train_df_remaining.lang == "nl") & (train_df_remaining.lang_rank <= want_nl)
    ]
    
    # Combine and sort
    train_df = (
        pd.concat([fr_take, nl_take], ignore_index=True)
        .sort_values("global_key")
        .reset_index(drop=True)
    )
    
    return train_df, val_df, fr_take, nl_take


# Load data
data_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/data")
train_df_full = pd.read_csv(data_dir / "train.csv")

# Test with same seed, different proportions
seed = 42
props = [0.2, 0.5, 0.8]

results = {}
for prop in props:
    train_df, val_df, fr_take, nl_take = simulate_sampling(train_df_full, prop, seed)
    results[prop] = {
        'train': train_df,
        'val': val_df,
        'fr_sentences': set(fr_take['input'].values),
        'nl_sentences': set(nl_take['input'].values),
        'train_sentences': set(train_df['input'].values),
        'val_sentences': set(val_df['input'].values)
    }

print("=" * 80)
print("TEST 1: Validation Set Consistency")
print("=" * 80)
# All proportions should have IDENTICAL validation sets
val_02 = results[0.2]['val_sentences']
val_05 = results[0.5]['val_sentences']
val_08 = results[0.8]['val_sentences']

print(f"Validation size at p=0.2: {len(val_02)}")
print(f"Validation size at p=0.5: {len(val_05)}")
print(f"Validation size at p=0.8: {len(val_08)}")
print(f"✅ Same validation? {val_02 == val_05 == val_08}")

print("\n" + "=" * 80)
print("TEST 2: Nested Sampling - French")
print("=" * 80)
# French at p=0.5 should CONTAIN ALL French from p=0.2
fr_02 = results[0.2]['fr_sentences']
fr_05 = results[0.5]['fr_sentences']
fr_08 = results[0.8]['fr_sentences']

print(f"French sentences at p=0.2: {len(fr_02)}")
print(f"French sentences at p=0.5: {len(fr_05)}")
print(f"French sentences at p=0.8: {len(fr_08)}")
print(f"✅ p=0.5 contains all from p=0.2? {fr_02.issubset(fr_05)}")
print(f"✅ p=0.8 contains all from p=0.5? {fr_05.issubset(fr_08)}")

if not fr_02.issubset(fr_05):
    print(f"❌ FAILED: p=0.5 missing {len(fr_02 - fr_05)} sentences from p=0.2")

print("\n" + "=" * 80)
print("TEST 3: Nested Sampling - Dutch")
print("=" * 80)
# Dutch at p=0.2 should CONTAIN ALL Dutch from p=0.5 (inverse relationship!)
nl_02 = results[0.2]['nl_sentences']
nl_05 = results[0.5]['nl_sentences']
nl_08 = results[0.8]['nl_sentences']

print(f"Dutch sentences at p=0.2: {len(nl_02)}")
print(f"Dutch sentences at p=0.5: {len(nl_05)}")
print(f"Dutch sentences at p=0.8: {len(nl_08)}")
print(f"✅ p=0.2 contains all from p=0.5? {nl_05.issubset(nl_02)}")
print(f"✅ p=0.5 contains all from p=0.8? {nl_08.issubset(nl_05)}")

if not nl_05.issubset(nl_02):
    print(f"❌ FAILED: p=0.2 missing {len(nl_05 - nl_02)} sentences from p=0.5")

print("\n" + "=" * 80)
print("TEST 4: Ordering Consistency")
print("=" * 80)
# For overlapping sentences, order should be the same
train_02 = results[0.2]['train']
train_05 = results[0.5]['train']

# Get common sentences
common_sentences = results[0.2]['train_sentences'] & results[0.5]['train_sentences']
print(f"Common sentences between p=0.2 and p=0.5: {len(common_sentences)}")

# Get their order in each dataset
order_02 = train_02[train_02['input'].isin(common_sentences)]['input'].tolist()
order_05 = train_05[train_05['input'].isin(common_sentences)]['input'].tolist()

print(f"✅ Same order for common sentences? {order_02 == order_05}")

if order_02 != order_05:
    # Find first mismatch
    for i, (s1, s2) in enumerate(zip(order_02, order_05)):
        if s1 != s2:
            print(f"❌ First mismatch at position {i}")
            print(f"   p=0.2: {s1[:50]}...")
            print(f"   p=0.5: {s2[:50]}...")
            break

print("\n" + "=" * 80)
print("TEST 5: Language Interleaving")
print("=" * 80)
# Check that languages are interleaved, not blocked
train_05 = results[0.5]['train']
lang_sequence = train_05['lang'].tolist()[:100]  # First 100 sentences

# Count transitions
transitions = sum(1 for i in range(len(lang_sequence)-1) 
                  if lang_sequence[i] != lang_sequence[i+1])

print(f"Language transitions in first 100 sentences: {transitions}")
print(f"✅ Well interleaved? {transitions > 20}")  # Should have many transitions

# Show pattern
print(f"Pattern: {''.join(['F' if l=='fr' else 'N' for l in lang_sequence[:50]])}")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
all_tests_passed = (
    val_02 == val_05 == val_08 and
    fr_02.issubset(fr_05) and
    fr_05.issubset(fr_08) and
    nl_05.issubset(nl_02) and
    nl_08.issubset(nl_05) and
    order_02 == order_05 and
    transitions > 20
)

if all_tests_passed:
    print("✅ ALL TESTS PASSED! Nested sampling is working correctly.")
else:
    print("❌ SOME TESTS FAILED! Check the output above.")

TEST 1: Validation Set Consistency
Validation size at p=0.2: 12950
Validation size at p=0.5: 12950
Validation size at p=0.8: 12950
✅ Same validation? True

TEST 2: Nested Sampling - French
French sentences at p=0.2: 24630
French sentences at p=0.5: 61577
French sentences at p=0.8: 98523
✅ p=0.5 contains all from p=0.2? True
✅ p=0.8 contains all from p=0.5? True

TEST 3: Nested Sampling - Dutch
Dutch sentences at p=0.2: 98524
Dutch sentences at p=0.5: 61577
Dutch sentences at p=0.8: 24631
✅ p=0.2 contains all from p=0.5? True
✅ p=0.5 contains all from p=0.8? True

TEST 4: Ordering Consistency
Common sentences between p=0.2 and p=0.5: 86207
✅ Same order for common sentences? True

TEST 5: Language Interleaving
Language transitions in first 100 sentences: 51
✅ Well interleaved? True
Pattern: FFNNNNFFNNFNNNFNNNNFFFFNNNNFNNFFFFNFFFFNNFFFFFFNFF

SUMMARY
✅ ALL TESTS PASSED! Nested sampling is working correctly.
